<a href="https://colab.research.google.com/github/sokrypton/7.571/blob/main/L8/quick_intro_to_jax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Quick Intro to JAX

JAX = NumPy + automatic differentiation + GPU/TPU

Three things to know:
1. `jax.numpy` works like `numpy`
2. `jax.grad` gives you gradients of any function
3. That's enough to do gradient descent!

---

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

## 1. jax.numpy ≈ numpy

In [ ]:
np.zeros(5)

In [ ]:
jnp.zeros(5)

Same API, but JAX arrays can run on GPU/TPU.

---
## 2. jax.grad — automatic gradients

Write a function, get its derivative for free.

In [ ]:
def square(x):
    return jnp.square(x)

square(10.0)

In [ ]:
# d/dx of x² = 2x
grad_square = jax.grad(square)
grad_square(10.0)  # should be 20.0

---
## 3. Example: Learn y = mx + b

Let's use `jax.grad` to fit a line to data via gradient descent.

In [ ]:
# Generate data: true m=2, true b=5
x = np.random.normal(size=10)
y = 2.0 * x + 5.0

In [ ]:
def loss_fn(m, b, x, y):
    """Mean squared error."""
    pred = m * x + b
    return jnp.mean(jnp.square(pred - y))

In [ ]:
# Start with a bad guess
m, b = 0.0, 0.0

plt.scatter(x, y, label='data')
plt.plot([-2, 2], [m*-2+b, m*2+b], 'r-', label=f'fit: m={m:.1f}, b={b:.1f}')
plt.legend(); plt.xlabel('X'); plt.ylabel('Y')
plt.title('Before training')
plt.show()

print(f'Loss: {loss_fn(m, b, x, y):.2f}')

In [ ]:
# jax.grad gives us gradients w.r.t. m and b (arguments 0 and 1)
grad_fn = jax.grad(loss_fn, argnums=(0, 1))

In [ ]:
# One gradient step
dm, db = grad_fn(m, b, x, y)
print(f'Gradients:  dm={dm:.4f}  db={db:.4f}')
print('These tell us which direction to nudge m and b to reduce loss.')

In [ ]:
# Training loop — just repeat: compute gradient, update params
m, b = 0.0, 0.0
lr = 0.1

for i in range(50):
    dm, db = grad_fn(m, b, x, y)
    m = m - lr * dm
    b = b - lr * db

print(f'Learned:  m={m:.4f}  b={b:.4f}')
print(f'True:     m=2.0000  b=5.0000')

In [ ]:
plt.scatter(x, y, label='data')
plt.plot([-2, 2], [m*-2+b, m*2+b], 'r-', label=f'fit: m={m:.2f}, b={b:.2f}')
plt.legend(); plt.xlabel('X'); plt.ylabel('Y')
plt.title('After training')
plt.show()

---
## That's it!

The pattern for everything we'll do in JAX:

```python
# 1. Define a loss function
def loss_fn(params, data):
    ...
    return loss

# 2. Get gradients
grads = jax.grad(loss_fn)(params, data)

# 3. Update params
params = params - lr * grads

# 4. Repeat
```